# Appendix A — Plugging GEODE-RAG into a governed agent

The capstone (Chapter 16) assembled a full GEODE-RAG system over the
Northwind Industries annual report. This appendix is the **bridge** to the
agent book, *Beyond Prompt and Pray*: we wrap that pipeline as a single
typed, gated tool — `search_report` — and run it inside a minimal governed
executor, the same shape as the agent book's `search_policy`.

Everything below is grounded in `data/corpus_facts.md`. The only
GPU/Qwen-touching cell is the store reload + the real-Qwen `RagConfig`,
which the lead runs in CI; the deterministic path uses a scripted fake
`LLMBackend` so the notebook self-checks without a model.

In [ ]:
import os, sys
KNOWLYTIX_SRC = os.environ.get("KNOWLYTIX_SRC", "/home/asudjianto/jupyterlab/GMS-knowlytix")
sys.path.insert(0, KNOWLYTIX_SRC)

## A.1 A scripted backend for deterministic CI

The pipeline needs an `LLMBackend` for query-triple extraction and grounded
synthesis. For a reproducible notebook we script one: it answers the single
question this appendix asks, returning the canonical query triple and an
answer that quotes only the retrieved fact. The real Qwen path is shown in
§A.5. The interface is the real one — `knowlytix.knowledge.llm_backend.LLMBackend`.

In [ ]:
from knowlytix.knowledge.llm_backend import LLMBackend


class ScriptedBackend(LLMBackend):
    """Deterministic stand-in for Qwen so the bridge self-checks offline.

    extract: maps the appendix's question to its canonical query triple
             (cloud platform, has_revenue, ?). synthesize: echoes the one
             retrieved figure verbatim — no parametric memory, no other number.
    """

    @property
    def model_name(self) -> str:
        return "scripted-fake"

    def call(self, system: str, user: str, max_tokens: int = 2048) -> str:
        low = (system + " " + user).lower()
        if "query triple" in low or "extract" in low or "relation" in low:
            return "cloud platform | has_revenue | ?"
        # synthesis: answer ONLY from the evidence block we were handed
        return "Cloud Platform reported revenue of 120.0 for FY2025."

## A.2 The pipeline behind the tool

We reload the store F2 trained and wrap it bank-grade: dense fallback off,
strict mode on, self-verification on. This is the only cell that touches the
store on disk — the lead runs it in CI. (`RagConfig` and `RagPipeline` are the
real library symbols from `knowlytix.knowledge.rag`.)

In [ ]:
# CI-ONLY: loads the trained store + builds the pipeline. Skipped at authoring
# time (one shared GPU). The bridge logic below does not depend on this cell.
import torch
from knowlytix.knowledge.query import DocGMSConfig, GMSExpertStore
from knowlytix.knowledge.rag import RagConfig, RagPipeline

STORE_PATH = os.environ.get("GMS_RAG_STORE", os.path.join(
    os.path.join(os.path.dirname(os.getcwd()), "code") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd(),
    "data", "gms_annual_report_store"))


def build_pipeline(llm: LLMBackend) -> RagPipeline:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    config = DocGMSConfig(store_path=STORE_PATH, ingest_mode="regex")
    store = GMSExpertStore(config, device=device)
    if not store.load():
        raise RuntimeError(f"no store at {STORE_PATH}; run scripts/build_store.py")
    rag = RagConfig(
        llm=llm,
        binding="embedding",      # paraphrases bind (Chapter 7)
        dense_fallback=False,      # quarantined dense index never used
        strict_mode=True,          # graph-only; abstain rather than guess
        verify_llm_output=True,    # GMS self-verification (Chapter 10)
        on_verify_fail="abstain",  # a contradicted claim never ships
    )
    return RagPipeline.from_store(store, rag)

## A.3 The tool: typed in, typed out, provenance carried

The agent book registers tools as typed records with pydantic input/output
schemas and a risk level (agent book Ch 6). We mirror that here without
importing the private `agentlab` package — the schema *is* the contract.

The key design choice: the tool's output carries the answer **and** its
provenance (the `file:line:char` span and the raw source text) **and** the
decision/verified flags. An agent that calls this tool gets a citable,
abstention-aware result, not a bare string. When the pipeline abstains the
tool returns `decision="abstain"` and an empty `sources` list — the agent
can branch on that instead of fabricating.

In [ ]:
from pydantic import BaseModel, Field


class SearchReportInput(BaseModel):
    question: str = Field(..., description="A natural-language question about the annual report.")


class Citation(BaseModel):
    triple: tuple[str, str, str]
    location: str | None = None   # file:line:char span
    raw: str | None = None        # the source text the fact came from


class SearchReportOutput(BaseModel):
    answer: str
    decision: str                 # accept | abstain | escalate
    verified: bool                # GMS-verified, not LLM-judged
    confidence: float
    route: str                    # triple | dense_fallback
    sources: list[Citation] = Field(default_factory=list)
    notice: str | None = None

In [ ]:
def search_report(pipe: RagPipeline, question: str) -> SearchReportOutput:
    """Answer a question through GEODE-RAG; return a typed, cited result.

    The pipeline does the work (extract -> bind -> retrieve -> synthesize ->
    verify -> decide). This wrapper only adapts RagAnswer into the agent's
    tool-output schema, preserving provenance and the abstain decision.
    """
    ans = pipe.query(question)
    citations = [
        Citation(triple=(f.head, f.relation, f.tail), location=f.location, raw=f.raw)
        for f in ans.sources
    ]
    return SearchReportOutput(
        answer=ans.answer,
        decision=ans.decision,
        verified=ans.verified,
        confidence=ans.confidence,
        route=ans.route,
        sources=citations,
        notice=ans.notice,
    )

## A.4 A minimal governed loop

The agent book runs every tool call through a `GovernedToolExecutor`: gates
fire first (schema, policy, plausibility), then the tool runs, and the result
is recorded for the audit trail (agent book Ch 6, Ch 15). We reproduce the
*shape* in a few lines so the appendix stands alone; the production executor
with its full gate protocol lives in the agent book — we do not duplicate it.

Two gates suffice to show the contract: a **syntax** gate that validates the
arguments against the tool's input schema, and a **risk** gate that records
the tool's risk level. A real deployment adds policy gates (agent book Ch 6).

In [ ]:
from dataclasses import dataclass, field
from typing import Any, Callable


@dataclass(frozen=True)
class Tool:
    """Typed, gated tool record — the agent book's Tool contract, minimal form."""
    name: str
    description: str
    input_schema: type[BaseModel]
    fn: Callable[..., BaseModel]
    risk: str = "low"


@dataclass
class ToolResult:
    tool_name: str
    output: Any = None
    error: str | None = None
    allowed: bool = False
    gates: list[str] = field(default_factory=list)


class GovernedExecutor:
    """Validate args against the schema, record risk, then run the tool.

    Mirrors agent book GovernedToolExecutor.execute: gates first, tool second,
    everything captured for the audit trail. A deny short-circuits before fn.
    """

    def __init__(self, tools: dict[str, Tool]):
        self._tools = tools
        self.audit: list[dict] = []

    def execute(self, name: str, arguments: dict) -> ToolResult:
        tool = self._tools.get(name)
        if tool is None:
            return ToolResult(name, error=f"unknown tool {name!r}")
        try:
            parsed = tool.input_schema(**arguments)  # syntax gate
        except Exception as e:
            return ToolResult(name, error=f"schema: {e}", gates=["syntax:deny"])
        gates = ["syntax:allow", f"risk:{tool.risk}"]
        output = tool.fn(**parsed.model_dump())
        result = ToolResult(name, output=output, allowed=True, gates=gates)
        self.audit.append({
            "tool": name, "gates": gates,
            "decision": getattr(output, "decision", None),
            "verified": getattr(output, "verified", None),
        })
        return result

Register `search_report` and call it through the executor. We bind the
pipeline into the tool's `fn` so the registry holds a zero-argument-besides-
schema callable, exactly as the agent book wires a stateful tool.

In [ ]:
# CI-ONLY: needs the real pipeline. The deterministic self-check (§A.6) injects
# a fake pipeline so this same code runs offline.
def make_executor(pipe: RagPipeline) -> GovernedExecutor:
    tool = Tool(
        name="search_report",
        description="Search the Northwind FY2025 annual report via GEODE-RAG; returns a grounded, cited answer or abstains.",
        input_schema=SearchReportInput,
        fn=lambda question: search_report(pipe, question),
        risk="low",
    )
    return GovernedExecutor({"search_report": tool})

## A.5 The real Qwen path (CI)

Everything above runs on the scripted backend for determinism. In production
the synthesis/extraction LLM is local Qwen2.5-3B-Instruct — no API key. Swap
the backend and the bridge is unchanged. The lead runs this in CI; we show it
as a listing here (one GPU is shared across authors).

In [ ]:
# CI-ONLY (real Qwen). Build the pipeline with a local Transformers backend
# and run the governed tool end to end.
def run_with_qwen(question: str) -> ToolResult:
    from knowlytix.knowledge.llm_backend import LocalTransformersBackend
    qwen = LocalTransformersBackend("Qwen/Qwen2.5-3B-Instruct")
    pipe = build_pipeline(qwen)
    executor = make_executor(pipe)
    return executor.execute("search_report", {"question": question})


# result = run_with_qwen("What was Cloud Platform's revenue?")
# print(result.output.answer, result.output.sources[0].location)

## A.6 Self-check — the tool returns a grounded, cited answer through the executor

We prove the chapter's claim without a GPU by injecting a **fake pipeline**
that returns the corpus-true fact for the canonical question:
`('cloud platform', 'has_revenue', '120.0')` at span `:15:553-558`
(see `data/corpus_facts.md`). The fake exposes the same `.query()` ->
`RagAnswer` contract the real `RagPipeline` does, so the tool and executor
code paths are exercised verbatim. The assertion checks that the answer flows
through the governed executor **with** provenance and **without** losing the
verified/decision flags — the whole point of the bridge.

In [ ]:
from knowlytix.knowledge.rag.pipeline import RagAnswer
from knowlytix.knowledge.rag.retrieve import RetrievedFact

# Corpus-true fact (data/corpus_facts.md): cloud platform has_revenue 120.0.
_FACT = RetrievedFact(
    head="cloud platform", relation="has_revenue", tail="120.0",
    score=0.0, confidence=1.0, source="triple",
    location="annual_report.md:15:553-558", raw="Cloud Platform | Technology | 120.0 | 340",
)


class _FakePipeline:
    """Stands in for RagPipeline.query() so the bridge self-checks offline."""
    def query(self, question: str) -> RagAnswer:
        return RagAnswer(
            answer="Cloud Platform reported revenue of 120.0 for FY2025.",
            confidence=1.0, decision="accept", route="triple", verified=True,
            sources=[_FACT],
        )


_pipe = _FakePipeline()
_tool = Tool(
    name="search_report",
    description="GEODE-RAG over the annual report.",
    input_schema=SearchReportInput,
    fn=lambda question: search_report(_pipe, question),
    risk="low",
)
executor = GovernedExecutor({"search_report": _tool})

result = executor.execute("search_report", {"question": "What was Cloud Platform's revenue?"})
out = result.output

# 1. the call passed the governance gates and ran
assert result.allowed and result.error is None, result
assert "syntax:allow" in result.gates and "risk:low" in result.gates
# 2. the answer is grounded: it quotes the corpus figure, carries provenance,
#    and keeps the verified/decision flags through the executor
assert "120.0" in out.answer
assert out.decision == "accept" and out.verified is True and out.route == "triple"
assert out.sources, "the tool dropped provenance"
assert out.sources[0].triple == ("cloud platform", "has_revenue", "120.0")
assert out.sources[0].location == "annual_report.md:15:553-558"
# 3. the audit trail recorded the gated, verified call
assert executor.audit[-1]["verified"] is True
print("OK: search_report returned a grounded, cited answer through the governed executor")
print(out.answer, "<-", out.sources[0].location)

## A.7 Abstention crosses the bridge too

A governed agent must be able to tell *"I don't know"* from a wrong answer.
Because the pipeline abstains on a prose/blind-spot question
(Chapter 11; coverage blind spots in `data/corpus_facts.md`: MD&A, Risk
Factors, Outlook), the tool surfaces `decision="abstain"` with empty
`sources`. The agent branches on that instead of fabricating.

In [ ]:
class _AbstainPipeline:
    def query(self, question: str) -> RagAnswer:
        return RagAnswer(
            answer="I cannot answer this from the available grounded evidence.",
            confidence=0.0, decision="abstain", route="triple", verified=True,
            notice="Question did not bind to known entities/relations.",
        )


_abstain_tool = Tool(
    name="search_report", description="GEODE-RAG.",
    input_schema=SearchReportInput,
    fn=lambda question: search_report(_AbstainPipeline(), question),
)
abstain_exec = GovernedExecutor({"search_report": _abstain_tool})
ab = abstain_exec.execute("search_report", {"question": "What does the Outlook section say?"})
assert ab.output.decision == "abstain" and not ab.output.sources
print("OK: the tool abstains on a blind-spot question, carrying the notice:")
print(ab.output.notice)

## A.8 Exercise solution — a policy gate

The minimal executor has only a syntax gate and a risk tag. Here is a
subclass that adds a **policy** gate denying an empty or over-long question
*before* the tool runs — a crude prompt-injection / cost guard. It mirrors
the agent book's `Gate` protocol (Ch 6): check first, deny short-circuits.

In [ ]:
class PolicyGovernedExecutor(GovernedExecutor):
    """GovernedExecutor + a policy gate on the question argument."""

    MAX_LEN = 500

    def execute(self, name: str, arguments: dict) -> ToolResult:
        q = str(arguments.get("question", "")).strip()
        if not q or len(q) > self.MAX_LEN:
            res = ToolResult(name, error="policy: empty or over-long question",
                             allowed=False, gates=["policy:deny"])
            self.audit.append({"tool": name, "gates": ["policy:deny"],
                               "decision": None, "verified": None})
            return res
        return super().execute(name, arguments)


_pol = PolicyGovernedExecutor({
    "search_report": Tool(
        name="search_report", description="GEODE-RAG.",
        input_schema=SearchReportInput,
        fn=lambda question: search_report(_FakePipeline(), question),
    )
})
# a normal question still passes
ok = _pol.execute("search_report", {"question": "What was Cloud Platform's revenue?"})
assert ok.allowed and "120.0" in ok.output.answer
# an empty question is denied before the tool runs
bad = _pol.execute("search_report", {"question": "   "})
assert not bad.allowed and bad.gates == ["policy:deny"]
assert _pol.audit[-1]["gates"] == ["policy:deny"]
print("OK: policy gate denies an empty question, allows a real one")